# 🔥 Web Attack Detection v8 — DistilBERT 11-class WAF classifier (fresh train)

Viết lại từ đầu: **train fresh từ `distilbert-base-uncased`** (KHÔNG warm-start v7),
**3 epoch**, **dataset đầy đủ** (eval trên toàn bộ val 51,099 dòng).

11 lớp: `normal, sqli, xss, cmdi, path_traversal, ssrf, xxe, log4shell, ssti, nosqli, crlf`.
Format text khớp byte-for-byte hợp đồng canonical (`preprocess_canonical_v6.canonical_compose`).

Dataset: `data/processed_v8_canonical/{train,val,test}.csv` (train 408,801 / val 51,099 / test 51,099).

## Hướng dẫn
1. Build dataset v8 (local): `python augment_v8.py && python build_v8_dataset.py`.
2. Upload 3 file CSV lên `MyDrive/web_attack_detection_v8/`.
3. Runtime → GPU (A100/L4 khuyến nghị; T4 ~5–6h cho 3 epoch fresh).
4. Chạy từng cell từ trên xuống.

## CELL 1 — Cài package

In [ ]:
import subprocess, sys
for pkg in ["transformers", "datasets", "accelerate", "scikit-learn", "seaborn", "matplotlib"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import torch
print("✅ packages ready | CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## CELL 2 — Mount Drive + kiểm tra dữ liệu

In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/web_attack_detection_v8"
SAVE_DIR = os.path.join(DATA_DIR, "model_v8")
os.makedirs(SAVE_DIR, exist_ok=True)

for fn in ["train.csv", "val.csv", "test.csv"]:
    p = os.path.join(DATA_DIR, fn)
    if os.path.exists(p):
        n = sum(1 for _ in open(p, encoding="utf-8")) - 1
        print(f"  ✅ {fn:10s} {n:>8,d} rows")
    else:
        print(f"  ❌ {fn:10s} NOT FOUND — upload vào {DATA_DIR}")
print("💾 save →", SAVE_DIR)

## CELL 3 — Config

Train **fresh** từ `distilbert-base-uncased`. Không warm-start → LR chuẩn 2e-5, 3 epoch.

In [ ]:
LABEL2ID = {
    "normal": 0, "sqli": 1, "xss": 2, "cmdi": 3, "path_traversal": 4,
    "ssrf": 5, "xxe": 6, "log4shell": 7, "ssti": 8, "nosqli": 9,
    "crlf": 10,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)

BASE_MODEL   = "distilbert-base-uncased"  # fresh train, KHÔNG warm-start
MAX_LENGTH   = 256          # khớp tokenizer contract
EPOCHS       = 3.0          # fresh train cần nhiều epoch hơn warm-start
BATCH        = 32           # T4 ok; L4/A100 có thể 64
LR           = 2e-5         # LR chuẩn cho fine-tune fresh
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
LABEL_SMOOTH = 0.05
SEED         = 42
MAX_TRAIN    = 0            # >0 để debug nhanh một phần train
print("base model:", BASE_MODEL, "| epochs:", EPOCHS, "| lr:", LR)

## CELL 4 — Load data (dataset đầy đủ) + phân phối nhãn

In [ ]:
import numpy as np, pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict

def load(split):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{split}.csv"))
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    df = df.dropna(subset=["text", "label_id"]).copy()
    df["text"] = df["text"].astype(str)
    return df

train_df, val_df, test_df = load("train"), load("val"), load("test")

if MAX_TRAIN and MAX_TRAIN < len(train_df):
    train_df = train_df.sample(n=MAX_TRAIN, random_state=SEED).reset_index(drop=True)
    print(f"  train capped → {len(train_df):,d} (debug)")

print(f"train {len(train_df):,d} | val {len(val_df):,d} | test {len(test_df):,d}")
print("\ntrain dist:")
c = Counter(train_df["label_id"])
for lid in sorted(c):
    print(f"  {ID2LABEL[lid]:16s} {c[lid]:>8,d}")

ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "label"}), preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text", "label_id"]].rename(columns={"label_id": "label"}), preserve_index=False),
    "test":       Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "label"}), preserve_index=False),
})
print(ds)

## CELL 5 — Tokenize

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tok_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

ds = ds.map(tok_fn, batched=True, batch_size=1000, desc="tokenize", remove_columns=["text"])
# KHÔNG dùng ds.set_format('torch'): torch-formatter import torchvision.io.VideoReader
# (đã bị bỏ trên Colab) -> ImportError. Để default_data_collator của Trainer tự tensorize.
print("✅ tokenized | cols:", ds["train"].column_names)

## CELL 6 — Class weights (balanced)

`normal` ~35% → weight thấp; `crlf` ~2.4k dòng → weight cao để bù lệch. Tự động, không boost tay.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
y = np.array(ds["train"]["label"])
weights = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=y)
class_weights = torch.tensor(weights, dtype=torch.float32)
for i, w in enumerate(weights):
    print(f"  {ID2LABEL[i]:16s} {w:.4f}")

## CELL 7 — Model + Trainer

In [ ]:
import torch.nn as nn
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback,
                          default_data_collator)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *a, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **_):
        labels = inputs.pop("labels")
        out = model(**inputs)
        cw = self.class_weights.to(out.logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTH)(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = float((preds == labels).mean())
    f1s = []
    for cls in range(len(LABEL2ID)):
        tp = int(((preds == cls) & (labels == cls)).sum())
        fp = int(((preds == cls) & (labels != cls)).sum())
        fn = int(((preds != cls) & (labels == cls)).sum())
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append(2 * p * r / (p + r) if (p + r) else 0.0)
    return {"accuracy": acc, "macro_f1": float(np.mean(f1s))}

args = TrainingArguments(
    output_dir=os.path.join(SAVE_DIR, "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("✅ trainer ready | train", len(ds["train"]), "| eval(full val)", len(ds["validation"]))

## CELL 8 — Train

In [ ]:
import time
t0 = time.time()
trainer.train()
print(f"\n✅ train xong trong {(time.time()-t0)/60:.1f} phút")

## CELL 9 — Eval full val + test (per-class report + confusion matrix)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

val_metrics = trainer.evaluate(ds["validation"])
print("VAL :", {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})

test_metrics = trainer.evaluate(ds["test"])
print("TEST:", {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

pred = trainer.predict(ds["test"])
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids
print("\n" + classification_report(y_true, y_pred, target_names=LABELS, digits=4))

cm = confusion_matrix(y_true, y_pred)
cmn = cm / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(10, 8))
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("pred"); plt.ylabel("true"); plt.title("v8 test confusion (row-normalized)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "confusion_v8.png"), dpi=120); plt.show()

## CELL 10 — Lưu model + label_config.json vào Drive

In [ ]:
FINAL = os.path.join(SAVE_DIR, "final_model_v8")
os.makedirs(FINAL, exist_ok=True)
trainer.save_model(FINAL)
tokenizer.save_pretrained(FINAL)

cfg = {
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()},
    "model_version": "v8_fresh_11class",
    "base_model": BASE_MODEL,
    "warm_start": False,
    "trained_from": "distilbert-base-uncased (fresh)",
    "max_length": MAX_LENGTH,
    "input_format": "WAF canonical (preprocess_canonical_v6.canonical_compose); 1-layer url-decode",
    "train_size": int(len(train_df)),
    "epochs": EPOCHS, "learning_rate": LR, "warmup_ratio": WARMUP_RATIO,
    "label_smoothing": LABEL_SMOOTH,
    "class_weight": "balanced (auto, no manual boost)",
    "dataset": "v8 full: 11 classes (normal + 9 attacks + crlf), CSIC-style augmented",
    "val_metrics": {k: float(v) for k, v in val_metrics.items() if isinstance(v, float)},
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, float)},
}
with open(os.path.join(FINAL, "label_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ saved →", FINAL)
for fn in sorted(os.listdir(FINAL)):
    print(f"  {fn}  ({os.path.getsize(os.path.join(FINAL, fn))/1e6:.1f} MB)")